# Domain-Specific Fine-Tuning (Colab Optimized)

This notebook fine-tunes a Transformer encoder on the San Diego Municipal Code using a Masked Language Modeling (MLM) objective for domain adaptation.

**This notebook is recommended to run on Google Colab with a GPU (T4 or higher).**

### Objectives
1. **Tokenize the Legal Corpus**: Process the ~52k document chunks for model training.
2. **Masked Language Modeling (MLM)**: Perform domain adaptation for legal vocabulary.
3. **Model Export**: Save and package the fine-tuned weights for local RAG integration.

## 1. Environment Setup & Drive Mount

Initialize the training environment by installing necessary libraries and mounting Google Drive. This section handles the conditional setup for both Colab and local environments, ensuring that domain adaptation can proceed with access to large-scale storage for model weights.

In [1]:
# Install training libraries
!pip install transformers datasets accelerate -q

import torch
import os
import math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- DATA PATH CONFIGURATION ---
if IN_COLAB:
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    dataset_filename = "/content/drive/MyDrive/sd-land-use-rag/data/processed/san_diego_code_chunks.jsonl"
else:
    print("Running locally. Dataset path set to local storage.")
    dataset_filename = "../data/processed/san_diego_code_chunks.jsonl"

if os.path.exists(dataset_filename):
    print(f"SUCCESS: Dataset found at {dataset_filename}")
else:
    print(f"ERROR: Dataset NOT found at {dataset_filename}. Check your path!")


Using device: cuda
Mounting Google Drive...
Mounted at /content/drive
SUCCESS: Dataset found at /content/drive/MyDrive/sd-land-use-rag/data/processed/san_diego_code_chunks.jsonl


## 2. Dataset Processing

Load the pre-processed San Diego Municipal Code chunks and prepare them for Masked Language Modeling. This stage involves initializing a legal-specific tokenizer and mapping it across the corpus to create the tensor inputs required for Transformer training.

In [2]:
# 1. Load Dataset
dataset = load_dataset("json", data_files=dataset_filename, split="train")

# 2. Initialize Tokenizer (from Legal-BERT)
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=dataset.column_names
)

print(f"Total tokenized records: {len(tokenized_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizing dataset...


Map (num_proc=4):   0%|          | 0/52384 [00:00<?, ? examples/s]

Total tokenized records: 52384


## 3. MLM Training

Execute the Masked Language Modeling (MLM) objective. By masking a percentage of tokens and tasking the model with their reconstruction, we adapt the underlying BERT encoder to the specific linguistic patterns and legal citations unique to the San Diego Municipal Code.

In [4]:
# 1. Split the dataset for training and evaluation (90/10 split)
print("Splitting dataset into train and validation sets...")
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# 2. Initialize Model with weight-tying fix to silence warnings
model = AutoModelForMaskedLM.from_pretrained(
    model_name,
    tie_word_embeddings=False
).to(device)

# 3. Setup Data Collator (15% random masking)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# 4. Define Training Arguments
# Removed 'overwrite_output_dir' as it causes TypeErrors in some Colab environments
training_args = TrainingArguments(
    output_dir="sd_legal_bert_mlm",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    push_to_hub=False
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training loop...")
trainer.train()


Splitting dataset into train and validation sets...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting training loop...


Epoch,Training Loss,Validation Loss
1,0.896834,0.859901
2,0.751924,0.714888
3,0.761928,0.701327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8841, training_loss=0.8807686843587295, metrics={'train_runtime': 4132.457, 'train_samples_per_second': 34.225, 'train_steps_per_second': 2.139, 'total_flos': 1.185610741830144e+16, 'train_loss': 0.8807686843587295, 'epoch': 3.0})

## 4. Evaluation and Perplexity

Quantify the success of the domain adaptation by calculating the model's loss and perplexity on a held-out validation set. Perplexity measures how well the probability distribution predicted by the model matches the actual distribution of the text; a lower score indicates a more 'familiar' model.

In [5]:

print("Evaluating model performance...")
eval_results = trainer.evaluate()

loss = eval_results["eval_loss"]
perplexity = math.exp(loss)

print(f"\n--- Final Metrics ---")
print(f"Evaluation Loss: {loss:.4f}")
print(f"Perplexity:      {perplexity:.4f}")

Evaluating model performance...



--- Final Metrics ---
Evaluation Loss: 0.6786
Perplexity:      1.9712


## 5. Model Export

Persist the fine-tuned model weights and tokenizer configuration to permanent storage. Saving directly to Google Drive ensures the trained artifacts are preserved and can be easily transferred back to the local `models/` directory for RAG integration.

In [6]:
# 1. Save directly to Google Drive
save_dir = "/content/drive/MyDrive/sd-land-use-rag/models/san_diego_legal_bert"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model successfully saved to {save_dir}!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to /content/drive/MyDrive/sd-land-use-rag/models/san_diego_legal_bert!
